In [9]:
##2
# --- Complete Interactive Light Intensity Heatmap (Final) ---
# - Colorbar label always at top of the bar (both vertical/horizontal)
# - "Keep colorbar label horizontal" toggle works correctly
# - No black border option
# - Tabbed interface: Basic / Advanced

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.image as mpimg
import io
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Helper to decode uploaded file content ---
def get_file_content(file_dict):
    content = file_dict['content']
    if isinstance(content, memoryview):
        return content.tobytes()
    elif isinstance(content, bytes):
        return content
    else:
        return bytes(content)

# --- Custom blue → red colormap ---
blue_red_cmap = LinearSegmentedColormap.from_list("blue_red_only", ["blue", "purple", "red"])

# --- Available colormaps ---
cmap_options = {
    'Custom Blue-Red': blue_red_cmap,
    'Viridis': 'viridis',
    'Plasma': 'plasma',
    'Inferno': 'inferno',
    'Magma': 'magma',
    'Coolwarm': 'coolwarm',
    'RdBu': 'RdBu',
    'Jet': 'jet',
    'Rainbow': 'rainbow',
    'Turbo': 'turbo',
    'Cividis': 'cividis',
    'Spectral': 'Spectral'
}
cmap_keys = list(cmap_options.keys())

# ======================== WIDGETS ========================

# --- File uploads ---
csv_upload = widgets.FileUpload(description='Upload CSV', accept='.csv', multiple=False)
image_upload = widgets.FileUpload(description='Upload Image (optional)', accept='.png,.jpg,.jpeg', multiple=False)
use_image = widgets.Checkbox(description='Use background image', value=False)

# --- Plot bounds ---
xmin_input = widgets.FloatText(description='xmin', value=0.0)
xmax_input = widgets.FloatText(description='xmax', value=10.0)
ymin_input = widgets.FloatText(description='ymin', value=0.0)
ymax_input = widgets.FloatText(description='ymax', value=10.0)

# --- Image extent ---
img_xmin = widgets.FloatText(description='Image xmin', value=0.0)
img_xmax = widgets.FloatText(description='Image xmax', value=10.0)
img_ymin = widgets.FloatText(description='Image ymin', value=0.0)
img_ymax = widgets.FloatText(description='Image ymax', value=10.0)

# --- Labels and title ---
xlabel_input = widgets.Text(description='X label', value='Lateral Position (ft)')
ylabel_input = widgets.Text(description='Y label', value='Longitudinal Position (ft)')
title_input = widgets.Text(description='Plot title', value='Light Intensity Heatmap (Lux)')
cbar_label_input = widgets.Text(description='Colorbar label', value='Light Intensity (Lux)')

# --- Interpolation ---
interp_method = widgets.Dropdown(description='Interpolation:', options=['linear', 'nearest', 'cubic'], value='cubic')

# --- Colour scheme ---
color_scheme = widgets.Dropdown(description='Color scheme:', options=cmap_keys, value='Custom Blue-Red')

# --- Point labels ---
label_points = widgets.Checkbox(description='Label data points with intensity', value=False)

# ======================== ADVANCED CONTROLS ========================

opacity_slider = widgets.FloatSlider(description='Opacity:', min=0.0, max=1.0, step=0.05, value=0.75)
grid_resolution = widgets.IntSlider(description='Grid resolution:', min=50, max=500, step=50, value=200)

# X-label position: bottom or top
xlabel_pos = widgets.Dropdown(description='X-label position:', options=['bottom', 'top'], value='bottom')

# Y-label position: left or right
ylabel_pos = widgets.Dropdown(description='Y-label position:', options=['left', 'right'], value='left')

# Colorbar orientation and label behaviour
cbar_orientation = widgets.Dropdown(description='Colorbar orientation:', options=['vertical', 'horizontal'], value='vertical')
cbar_label_horizontal = widgets.Checkbox(description='Keep colorbar label horizontal', value=True)

# --- Plot button and output ---
plot_button = widgets.Button(description='Generate Plot', button_style='primary')
output_plot = widgets.Output()

# ======================== UI LAYOUT ========================

# Basic tab
basic_tab = widgets.VBox([
    widgets.HBox([csv_upload, image_upload]),
    widgets.HBox([use_image]),
    widgets.HTML("<b>Plot bounds (data coordinates):</b>"),
    widgets.HBox([xmin_input, xmax_input, ymin_input, ymax_input]),
    widgets.HTML("<b>Image extent (real‑world coordinates):</b>"),
    widgets.HBox([img_xmin, img_xmax, img_ymin, img_ymax]),
    widgets.HBox([interp_method, color_scheme]),
    widgets.HTML("<b>Labels and title:</b>"),
    widgets.HBox([xlabel_input, ylabel_input, title_input, cbar_label_input]),
    widgets.HBox([label_points])
])

# Advanced tab
advanced_tab = widgets.VBox([
    widgets.HBox([opacity_slider, grid_resolution]),
    widgets.HTML("<b>Label placement:</b>"),
    widgets.HBox([xlabel_pos, ylabel_pos]),
    widgets.HTML("<b>Colorbar settings:</b>"),
    widgets.HBox([cbar_orientation, cbar_label_horizontal])
])

# Main tab widget
tab = widgets.Tab([basic_tab, advanced_tab])
tab.set_title(0, 'Basic')
tab.set_title(1, 'Advanced')

# Final UI
ui = widgets.VBox([
    widgets.HTML("<h2>Light Intensity Heatmap Generator</h2>"),
    tab,
    plot_button,
    output_plot
])

# ======================== CALLBACKS ========================

def toggle_image_extents(change):
    visible = change['new']
    for w in [img_xmin, img_xmax, img_ymin, img_ymax]:
        w.layout.visibility = 'visible' if visible else 'hidden'
use_image.observe(toggle_image_extents, 'value')
toggle_image_extents({'new': use_image.value})

def generate_plot(b):
    with output_plot:
        clear_output(wait=True)

        # 1. Read CSV
        if not csv_upload.value:
            print("Please upload a CSV file first.")
            return
        try:
            uploaded_csv = list(csv_upload.value)[0]
        except IndexError:
            print("No file found in upload.")
            return
        content_bytes = get_file_content(uploaded_csv)
        df = pd.read_csv(io.BytesIO(content_bytes))
        if df.shape[1] < 3:
            print("CSV must contain at least three columns: x, y, z (intensity).")
            return
        x = df.iloc[:, 0].values
        y = df.iloc[:, 1].values
        z = df.iloc[:, 2].values

        # 2. Bounds
        xmin = xmin_input.value
        xmax = xmax_input.value
        ymin = ymin_input.value
        ymax = ymax_input.value

        # 3. Background image
        use_bg = use_image.value
        bg = None
        img_extent = None
        if use_bg:
            if not image_upload.value:
                print("Please upload an image file or uncheck 'Use background image'.")
                return
            uploaded_img = list(image_upload.value)[0]
            content_img_bytes = get_file_content(uploaded_img)
            bg = mpimg.imread(io.BytesIO(content_img_bytes))
            img_extent = (img_xmin.value, img_xmax.value, img_ymin.value, img_ymax.value)

        # 4. Interpolation
        method = interp_method.value
        res = grid_resolution.value
        grid_x, grid_y = np.mgrid[
            xmin:xmax:complex(0, res),
            ymin:ymax:complex(0, res)
        ]
        grid_z = griddata((x, y), z, (grid_x, grid_y), method=method)

        # 5. Colormap
        cmap = cmap_options[color_scheme.value]

        # 6. Create plot
        fig, ax = plt.subplots(figsize=(20, 17))

        if use_bg and bg is not None:
            ax.imshow(bg, extent=img_extent, origin='lower', alpha=1.0)

        # Heatmap
        im = ax.imshow(
            grid_z.T,
            extent=(xmin, xmax, ymin, ymax),
            origin='lower',
            cmap=cmap,
            alpha=opacity_slider.value
        )

        # 7. Colorbar – label always on top, horizontal if checked
        cbar_orient = cbar_orientation.value
        cbar = fig.colorbar(im, ax=ax, orientation=cbar_orient)

        label_text = cbar_label_input.value
        horizontal = cbar_label_horizontal.value

        # Always use x-axis, position top (valid for XAxis)
        cbar.ax.xaxis.set_label_position('top')
        cbar.ax.set_xlabel(label_text, rotation=0 if horizontal else 90)

        # 8. Axis labels and title with positions
        if xlabel_pos.value == 'top':
            ax.xaxis.set_label_position('top')
        else:
            ax.xaxis.set_label_position('bottom')
        ax.set_xlabel(xlabel_input.value)

        if ylabel_pos.value == 'right':
            ax.yaxis.set_label_position('right')
        else:
            ax.yaxis.set_label_position('left')
        ax.set_ylabel(ylabel_input.value)

        ax.set_title(title_input.value)

        # 9. No black border – keep default spines
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(True)
        ax.spines['left'].set_visible(True)

        # 10. Data point labels
        if label_points.value:
            for xi, yi, zi in zip(x, y, z):
                ax.text(xi, yi, f"{zi:.1f}", color="black", fontsize=7,
                        ha="center", va="center")

        plt.tight_layout()
        plt.show()

# --- Attach callback ---
plot_button.on_click(generate_plot)

# --- Display ---
display(ui)